# 02 - Feature Engineering

Technische Indikatoren & Feature-Erstellung für das ML-Modell.

**Ziele:**
- Technische Indikatoren berechnen (RSI, MACD, Bollinger Bands, etc.)
- Feature-Korrelationen analysieren
- Feature-Importance untersuchen
- Preprocessing-Pipeline testen

In [1]:
import structlog
structlog.configure(
    wrapper_class=structlog.make_filtering_bound_logger(0),
)

from backend.workflows.fetch_data import fetch_and_store

features_df, manifest = fetch_and_store()

print(f"Shape: {features_df.shape}")
print(f"Manifest Hash: {manifest.column_hash}")
print(f"Columns: {len(manifest.columns)}")
features_df.head()

/Users/e.autenrieth/Documents/GitHub/ml-stock-prediction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as erikautenrieth

Initialized MLflow to track repo "erikautenrieth/stock-prediction-hub"

Repository erikautenrieth/stock-prediction-hub initialized!

2026-03-08 19:41:15 [info     ] dagshub_initialized            experiment=sp500_prediction repo=erikautenrieth/stock-prediction-hub tracking_uri=https://dagshub.com/erikautenrieth/stock-prediction-hub.mlflow
2026-03-08 19:41:16 [info     ] dagshub_storage_initialized    bucket=stock-prediction-hub
2026-03-08 19:41:16 [info     ] data_store_remote_enabled     
2026-03-08 19:41:16 [info     ] downloading raw data           end=2026-03-08 start=2000-08-01 symbol=^GSPC


[*********************100%***********************]  1 of 1 completed

2026-03-08 19:41:16 [info     ] raw data downloaded            rows=6437
2026-03-08 19:41:16 [info     ] saved                          path=backend/data/raw/sp500_ohlcv.parquet rows=6437


2026-03-08 19:41:18 [info     ] uploaded                       local=backend/data/raw/sp500_ohlcv.parquet remote=raw/sp500_ohlcv.parquet
2026-03-08 19:41:18 [info     ] computing target              
2026-03-08 19:41:18 [info     ] computing indicators          
2026-03-08 19:41:18 [info     ] calc_indicators: 33 rows dropped (NaN), 6404 remaining
2026-03-08 19:41:18 [info     ] fetching extra market data    
2026-03-08 19:41:27 [info     ] fetch_extra_market_data: filled 513 NaN values, 0 remaining
2026-03-08 19:41:27 [info     ] transforming to returns       
2026-03-08 19:41:27 [info     ] transform_to_returns: 1 rows dropped (NaN from pct_change), 6403 remaining
2026-03-08 19:41:27 [info     ] features ready                 columns=53 hash=daca9010a42ea73c rows=6403
2026-03-08 19:41:27 [info     ] saved                          path=backend/data/features/sp500_features.parquet rows=6403
2026-03-08 19:41:28 [info     ] uploaded                       local=backend/data/features/sp500

Price,Close,Volume,Target,SMA 10,EMA 10,EMA 20,WMA 10,Momentum 10,SAR,RSI,...,^IBEX Close,SI=F Close,HG=F Close,NG=F Close,^TNX Close,^IRX Close,^FVX Close,^TYX Close,SPY Close,EFA Close
Date,,,,,,,,,,,,,,,,,,,,,
2000-09-19,0.010654,0.064831,0,0.013529,0.010990,0.015660,0.008156,-0.032317,0.025228,40.364915,...,-0.010958,0.001436,0.003266,0.014164,-0.021,0.02,0.007,-0.042,0.009073,0.0
2000-09-20,-0.005863,0.077178,0,0.016688,0.013871,0.019589,0.010555,-0.028188,0.025988,37.586141,...,-0.014839,-0.010854,-0.021704,-0.009311,0.042,0.00,0.046,0.048,-0.007386,0.0
2000-09-21,-0.001578,0.001268,0,0.014605,0.012660,0.019181,0.008826,-0.036893,0.021193,36.855211,...,-0.008981,0.008696,0.026622,-0.006203,-0.021,-0.01,-0.008,-0.035,-0.015206,0.0
2000-09-22,-0.000228,0.072462,0,0.011677,0.010547,0.017564,0.006358,-0.031600,0.016035,36.744288,...,0.020919,0.007800,-0.006483,-0.029506,-0.042,0.01,-0.040,-0.023,0.018178,0.0
2000-09-25,-0.006689,-0.171320,0,0.014998,0.014196,0.022091,0.009773,-0.034905,0.016614,33.551825,...,0.011759,0.001018,0.008157,0.028260,0.004,0.01,0.007,-0.013,-0.007098,0.0


## Datenqualität prüfen

In [2]:
import pandas as pd

nan_total = features_df.isna().sum().sum()
print(f"NaN gesamt: {nan_total}")
print(f"Duplikate: {features_df.index.duplicated().sum()}")
print(f"Zeitraum: {features_df.index.min()} → {features_df.index.max()}")
print(f"\nTarget-Verteilung:")
print(features_df["Target"].value_counts())
print(f"\nDtypes:")
print(features_df.dtypes.value_counts())

NaN gesamt: 0
Duplikate: 0
Zeitraum: 2000-09-19 00:00:00 → 2026-03-06 00:00:00

Target-Verteilung:
Target
1    3854
0    2549
Name: count, dtype: int64

Dtypes:
float64    52
int64       1
Name: count, dtype: int64


## Feature-Manifest validieren

In [3]:
errors = manifest.validate_against(features_df)
if errors:
    print("Manifest-Fehler:")
    for e in errors:
        print(f"  - {e}")
else:
    print(f"Manifest OK — {len(manifest.columns)} Spalten, Hash: {manifest.column_hash}")

print(f"\nSpalten:\n{manifest.columns}")

Manifest OK — 53 Spalten, Hash: daca9010a42ea73c

Spalten:
['%D', '%K', '%R', '+DMI', '-DMI', 'ADOSC', 'ADX', 'CCI', 'CL=F Close', 'CNY=X Close', 'Close', 'EFA Close', 'EMA 10', 'EMA 20', 'EURUSD=X Close', 'GBPUSD=X Close', 'GC=F Close', 'HG=F Close', 'JPY=X Close', 'MACD', 'MACD_HIST', 'MACD_SIGNAL', 'Momentum 10', 'NG=F Close', 'OBV', 'ROC', 'RSI', 'SAR', 'SI=F Close', 'SMA 10', 'SPY Close', 'Target', 'Volume', 'WMA 10', '^AXJO Close', '^BSESN Close', '^DJI Close', '^FCHI Close', '^FTSE Close', '^FVX Close', '^GDAXI Close', '^HSI Close', '^IBEX Close', '^IRX Close', '^IXIC Close', '^MXX Close', '^N225 Close', '^RUT Close', '^TNX Close', '^TYX Close', 'low_band', 'mid_band', 'up_band']


## DataStore Roundtrip testen

In [4]:
from backend.infra.database import get_data_store
from backend.core.features.manifest import FeatureManifest

store = get_data_store()

loaded = store.load_features()
print(f"Saved: {features_df.shape} → Loaded: {loaded.shape}")

loaded_manifest = FeatureManifest.from_dataframe(loaded)
assert loaded_manifest.column_hash == manifest.column_hash, "Column hash mismatch!"
print(f"Roundtrip OK — Hash matches: {loaded_manifest.column_hash}")
loaded

2026-03-08 19:41:28 [info     ] dagshub_storage_initialized    bucket=stock-prediction-hub
2026-03-08 19:41:28 [info     ] data_store_remote_enabled     
Saved: (6403, 53) → Loaded: (6403, 53)
Roundtrip OK — Hash matches: daca9010a42ea73c


,Close,Volume,Target,SMA 10,EMA 10,EMA 20,WMA 10,Momentum 10,SAR,RSI,...,^IBEX Close,SI=F Close,HG=F Close,NG=F Close,^TNX Close,^IRX Close,^FVX Close,^TYX Close,SPY Close,EFA Close
Date,,,,,,,,,,,,,,,,,,,,,
2000-09-19,0.010654,0.064831,0,0.013529,0.010990,0.015660,0.008156,-0.032317,0.025228,40.364915,...,-0.010958,0.001436,0.003266,0.014164,-0.021,0.020,0.007,-0.042,0.009073,0.000000
2000-09-20,-0.005863,0.077178,0,0.016688,0.013871,0.019589,0.010555,-0.028188,0.025988,37.586141,...,-0.014839,-0.010854,-0.021704,-0.009311,0.042,0.000,0.046,0.048,-0.007386,0.000000
2000-09-21,-0.001578,0.001268,0,0.014605,0.012660,0.019181,0.008826,-0.036893,0.021193,36.855211,...,-0.008981,0.008696,0.026622,-0.006203,-0.021,-0.010,-0.008,-0.035,-0.015206,0.000000
2000-09-22,-0.000228,0.072462,0,0.011677,0.010547,0.017564,0.006358,-0.031600,0.016035,36.744288,...,0.020919,0.007800,-0.006483,-0.029506,-0.042,0.010,-0.040,-0.023,0.018178,0.000000
2000-09-25,-0.006689,-0.171320,0,0.014998,0.014196,0.022091,0.009773,-0.034905,0.016614,33.551825,...,0.011759,0.001018,0.008157,0.028260,0.004,0.010,0.007,-0.013,-0.007098,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-02,0.000398,-0.088000,0,0.000335,0.001260,0.001850,0.001293,0.006605,-0.013908,48.647591,...,-0.026246,-0.047453,-0.018320,0.035327,0.086,0.012,0.109,0.065,0.000569,-0.019738
2026-03-03,-0.009444,0.059713,0,0.009482,0.008842,0.010316,0.009045,-0.003901,0.019934,43.035212,...,-0.045668,-0.060724,-0.020528,0.031757,0.008,0.005,0.010,0.004,-0.008814,-0.031075
2026-03-04,0.007756,-0.184709,0,0.001541,0.000881,0.002298,0.000967,-0.001719,0.011379,48.264266,...,0.024885,-0.003497,0.014116,-0.044859,0.024,0.000,0.037,0.014,0.007055,0.012888


## Feature-Korrelation mit Target

In [5]:
corr = features_df.corr()["Target"].drop("Target").sort_values(ascending=False)

print(corr)


Price
%R                0.053718
+DMI              0.053008
MACD_HIST         0.052745
RSI               0.051695
CCI               0.044033
ROC               0.040314
Momentum 10       0.038132
ADX               0.034472
ADOSC             0.026685
MACD              0.017561
OBV               0.016204
%D                0.015418
EURUSD=X Close    0.008204
GBPUSD=X Close    0.007445
^BSESN Close      0.002825
MACD_SIGNAL       0.001964
^IRX Close        0.001212
^AXJO Close      -0.001163
Volume           -0.002298
%K               -0.002432
NG=F Close       -0.003243
^GDAXI Close     -0.003656
^FTSE Close      -0.005017
^FCHI Close      -0.005916
CNY=X Close      -0.006959
^MXX Close       -0.007046
^DJI Close       -0.007774
^IXIC Close      -0.008262
SI=F Close       -0.010230
JPY=X Close      -0.010984
SPY Close        -0.012242
Close            -0.012505
^HSI Close       -0.013852
^IBEX Close      -0.014184
up_band          -0.014538
GC=F Close       -0.014830
^RUT Close       -0.01

## Statistik der Indikatoren

In [6]:
indicator_cols = [
    "RSI", "ROC", "%R", "MACD", "CCI", "ADX", "+DMI", "-DMI",
    "%K", "%D", "OBV", "ADOSC",
]
existing = [c for c in indicator_cols if c in features_df.columns]
features_df[existing].describe().round(2)

Price,RSI,ROC,%R,MACD,CCI,ADX,+DMI,-DMI,%K,%D,OBV,ADOSC
count,6403.00,6403.00,6403.00,6403.00,6403.00,6403.00,6403.00,6403.00,6403.00,6403.00,6403.00,6403.00
mean,54.08,0.30,-38.28,0.00,20.77,22.98,24.39,23.71,55.26,55.25,0.00,0.10
std,11.32,3.31,31.45,0.01,106.47,7.98,7.25,7.36,32.86,18.69,0.12,0.21
min,13.64,-25.88,-100.00,-0.11,-355.74,8.14,5.34,5.28,0.00,0.00,-4.24,-0.61
25%,45.88,-1.23,-65.47,-0.00,-63.60,16.98,18.93,18.41,25.14,42.02,-0.01,-0.05
50%,55.27,0.64,-30.56,0.00,44.54,21.72,24.53,22.89,58.95,55.78,0.00,0.10
75%,62.63,2.11,-9.38,0.01,104.64,27.71,29.59,28.27,86.12,68.72,0.01,0.25
max,86.69,21.64,-0.00,0.03,318.65,55.05,51.35,62.28,100.00,99.29,3.21,0.75


## Skalierungsanalyse: Wertebereiche der Features

In [7]:
import pandas as pd

exclude = ["Target"]
feat_df = features_df.drop(columns=exclude)

# Gruppierung der Features
price_cols = [c for c in feat_df.columns if any(k in c for k in ["Close", "SMA", "EMA", "WMA", "SAR", "up_band", "mid_band", "low_band"])]
bounded_cols = [c for c in feat_df.columns if c in ["RSI", "%R", "%K", "%D", "ADOSC", "ADX", "+DMI", "-DMI"]]
unbounded_cols = [c for c in feat_df.columns if c in ["MACD", "MACD_SIGNAL", "MACD_HIST", "CCI", "ROC", "Momentum 10"]]
volume_cols = [c for c in feat_df.columns if c in ["Volume", "OBV"]]
extra_market_cols = [c for c in feat_df.columns if "Close" in c and c != "Close"]

groups = {
    "Preis-basiert (absolut)": price_cols,
    "Beschränkt (bounded)": bounded_cols,
    "Unbeschränkt (unbounded)": unbounded_cols,
    "Volumen": volume_cols,
    "Extra-Marktdaten": extra_market_cols,
}

for name, cols in groups.items():
    if not cols:
        continue
    subset = feat_df[cols]
    stats = pd.DataFrame({
        "Min": subset.min(),
        "Max": subset.max(),
        "Mean": subset.mean(),
        "Std": subset.std(),
        "Range": subset.max() - subset.min(),
    }).round(2)
    print(f"\n{'='*60}")
    print(f" {name} ({len(cols)} Features)")
    print(f"{'='*60}")
    print(stats.to_string())

# Gesamtübersicht: Faktor zwischen größtem und kleinstem Range
ranges = (feat_df.max() - feat_df.min())
print(f"\n{'='*60}")
print(f" Skalierungsproblem-Übersicht")
print(f"{'='*60}")
print(f"Kleinster Range:  {ranges.min():.2f} ({ranges.idxmin()})")
print(f"Größter Range:    {ranges.max():.0f} ({ranges.idxmax()})")
print(f"Verhältnis:       {ranges.max() / ranges.min():.0f}x")
print(f"\nTop 5 größte Ranges:")
print(ranges.nlargest(5).to_string())
print(f"\nTop 5 kleinste Ranges:")
print(ranges.nsmallest(5).to_string())


 Preis-basiert (absolut) (36 Features)
                 Min   Max  Mean   Std  Range
Price                                        
Close          -0.12  0.12  0.00  0.01   0.24
SMA 10         -0.08  0.19 -0.00  0.02   0.27
EMA 10         -0.06  0.16 -0.00  0.02   0.23
EMA 20         -0.07  0.23 -0.00  0.02   0.30
WMA 10         -0.07  0.14 -0.00  0.01   0.21
SAR            -0.17  0.31 -0.01  0.04   0.48
up_band        -0.02  0.50  0.03  0.04   0.52
mid_band       -0.10  0.26 -0.00  0.03   0.36
low_band       -0.22  0.06 -0.03  0.03   0.27
GC=F Close     -0.11  0.09  0.00  0.01   0.20
CL=F Close     -3.06  0.38 -0.00  0.05   3.44
EURUSD=X Close -0.13  0.17  0.00  0.01   0.31
GBPUSD=X Close -0.08  0.04 -0.00  0.01   0.11
JPY=X Close    -0.16  0.18  0.00  0.01   0.33
CNY=X Close    -0.09  0.10 -0.00  0.00   0.19
^IXIC Close    -0.12  0.14  0.00  0.02   0.26
^DJI Close     -0.13  0.11  0.00  0.01   0.24
^RUT Close     -0.14  0.09  0.00  0.02   0.24
^FTSE Close    -0.11  0.10  0.00  0.01  